# Clase 6 (sincrónica) · Datos georreferenciados y mapas

**Python para Economistas · UNLP**

### Plan de la clase (120 minutos)

| Minutos | Bloque | Quién trabaja |
| --- | --- | --- |
| 0 a 8 | Bienvenida, `git pull` y devoluciones del TP5 | Docente |
| 8 a 25 | **1. Calentamiento**: lo que cambió desde los videos y el cuestionario | Todos |
| 25 a 45 | **2. Del censo al mapa**: un `merge` verificado y las decisiones de un mapa | Docente y alumnos |
| 45 a 50 | Recreo | |
| 50 a 85 | **3. Desafío en salas**: ¿en qué provincia está cada localidad? | Equipos de 3 |
| 85 a 105 | **4. Departamentos**: códigos, `sjoin`, `dissolve` y mapas interactivos | Docente y alumnos |
| 105 a 120 | **5. Presentación del TP Final** | Docente |

**Objetivos.** Al terminar la clase van a poder:
1. leer capas geográficas con GeoPandas 1.x y diagnosticar los errores más comunes (archivos que faltan, código viejo);
2. unir datos económicos con una capa geográfica **por código**, y verificar la unión;
3. tomar (y justificar) las decisiones de un mapa: qué variable, qué proyección, qué escala, qué colores y qué hacer con los faltantes;
4. unir tablas **por ubicación** (`sjoin`, `sjoin_nearest`) y usar esa unión para auditar otra fuente;
5. combinar expresiones regulares, pandas y mapas en un mismo flujo, que es lo que les va a pedir el TP Final.

**Sobre el uso de IA en esta clase.** En las salas trabajen **sin asistentes**: las pistas están en el notebook y nosotros pasamos por las salas. Después de la clase pueden usar un asistente para pedir explicaciones, no soluciones.

## 0. Preparación

### 0.1 Instalación

El notebook asincrónico pedía instalar GeoPandas 0.14. **Ya no hace falta**: este notebook usa GeoPandas 1.x. En la terminal, con el entorno del curso activado:

```bash
conda install -c conda-forge geopandas mapclassify folium
# o bien
pip install geopandas mapclassify folium
```

En Colab, en una celda: `%pip install mapclassify folium` (GeoPandas ya viene instalado). Agreguen los tres paquetes al `requirements.txt` de sus proyectos.

### 0.2 Importaciones

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import mapclassify

print("geopandas", gpd.__version__)
print("pandas   ", pd.__version__)

# Este notebook usa funciones de GeoPandas 1.x (por ejemplo, union_all)
assert int(gpd.__version__.split(".")[0]) >= 1, "Actualicen GeoPandas: pip install -U geopandas"

### 0.3 Datos

Todos los archivos están en la carpeta `datos/`:

| Archivo | Qué es | Fuente |
| --- | --- | --- |
| `provincia.zip` | Provincias (GeoJSON comprimido) | IGN, capas SIG. Es el mismo archivo de los videos |
| `departamento.zip` | Departamentos y partidos (Shapefile comprimido) | IGN, capas SIG |
| `poblacion_censos.csv` | Población por provincia, censos 2010 y 2022 | INDEC (2022: resultados definitivos) |
| `localidades_ne.csv` | 158 localidades argentinas con coordenadas | Natural Earth (dominio público), *populated places* 1:10 millones |
| `ne_110m_admin_0_countries.geojson` | Países del mundo, escala 1:110 millones | Natural Earth. Reemplaza a `naturalearth_lowres` |
| `ne_50m_populated_places_simple.geojson` | Ciudades del mundo, escala 1:50 millones | Natural Earth. Reemplaza a `naturalearth_cities` |

In [ ]:
DATOS = Path("datos")
sorted(p.name for p in DATOS.iterdir())

---
## 1. Calentamiento: lo que cambió desde los videos

### 1.1 Qué cambió

| En los videos o el notebook asincrónico | Qué pasa hoy | Qué usar |
| --- | --- | --- |
| `gpd.datasets.get_path("naturalearth_lowres")` | Da `AttributeError`: los datos de ejemplo se eliminaron en GeoPandas 1.0 | Leer los archivos de Natural Earth de `datos/` |
| Instalar `geopandas==0.14.4` | Ya no hace falta, y muchas funciones nuevas no existen en esa versión | `geopandas` 1.x, con `mapclassify` y `folium` |
| La columna del país es `name` | En los archivos de Natural Earth las columnas están en mayúsculas | `NAME`, `CONTINENT`, `POP_EST` |
| «Las coordenadas del GeoJSON están en latitud y longitud» | Es al revés: **longitud y latitud**, es decir, `(x, y)` | `gpd.points_from_xy(longitud, latitud)` |
| El shapefile se nombra como un archivo | Es un **conjunto** de archivos: `.shp`, `.shx`, `.dbf`, `.prj` (y a veces `.cpg`). Si falta uno, falla o pierde información | Leer el `.zip` completo o usar GeoPackage |
| `.cx` para quedarse con una zona | `.cx` **selecciona filas**, no recorta geometrías | `.clip()` para recortar |
| `.area` en la capa original | En grados da números sin sentido, con un aviso | Reproyectar con `to_crs()` a una proyección de **igual área** |
| `read_file(..., crs="EPSG:4326")` | `read_file` no usa ese argumento: el CRS lo trae el archivo | `.crs` para consultarlo y `set_crs()` solo si falta |
| `sjoin(..., op="within")`, `unary_union` | Nombres viejos, eliminados o deprecados | `predicate="within"`, `union_all()` |

**Error 1: los datos de ejemplo.** La línea del notebook asincrónico:

In [ ]:
try:
    gpd.datasets.get_path("naturalearth_lowres")
except AttributeError as error:
    print("AttributeError:", error)

**Error 2: el shapefile incompleto.** Simulamos lo que pasa cuando se sube a Colab o se copia solo el archivo `.shp`:

In [ ]:
import tempfile
import zipfile

with zipfile.ZipFile(DATOS / "departamento.zip") as z:
    print(z.namelist())
    carpeta = Path(tempfile.mkdtemp())
    z.extract("departamento.shp", carpeta)   # extraemos SOLO el .shp

try:
    gpd.read_file(carpeta / "departamento.shp")
except Exception as error:
    print(type(error).__name__, ":", error)

El mensaje pide el `.shx`, que es el índice. Pero aunque lo reconstruyéramos, faltaría el `.dbf` (los nombres y códigos de los departamentos) y el `.prj` (el sistema de coordenadas). La solución es leer el `.zip` completo: GeoPandas lo abre sin descomprimirlo.

Fíjense en otro detalle de la lista: **no hay `.cpg`**, el archivo que dice en qué codificación están los textos. Lo retomamos en el bloque 4.

In [ ]:
departamentos = gpd.read_file(DATOS / "departamento.zip")
departamentos.shape

### 1.2 Pregunta 1 del cuestionario, con GeoPandas 1.x

> ¿Qué código escribirían para leer el archivo de ciudades del mundo y quedarse con las que están en Argentina?

Primero leemos las dos capas y miramos los nombres de las columnas.

In [ ]:
mundo = gpd.read_file(DATOS / "ne_110m_admin_0_countries.geojson")
ciudades = gpd.read_file(DATOS / "ne_50m_populated_places_simple.geojson")

print(mundo.shape, ciudades.shape)
print([col for col in mundo.columns if col in ("name", "NAME", "CONTINENT", "POP_EST")])

**Predecir antes de ejecutar.** Hay dos maneras de responder la pregunta: por ubicación (las ciudades que caen dentro del polígono de Argentina) y por atributo (la columna `adm0name`, que dice el país de cada ciudad).

✍️ ¿Van a dar el mismo número? Si no, ¿cuál da más, y por qué?

In [ ]:
# Escriban su código acá

In [ ]:
# Hagan un mapa con el polígono de Argentina y las ciudades que faltan en otro color

### 1.3 Pregunta 2 del cuestionario

> El área en km² se puede calcular como `mundo.to_crs({"proj": "cea"}).area / 10**6`. ¿Qué código escribirían para pintar cada país según su densidad de población?

`{"proj": "cea"}` es la proyección cilíndrica de **igual área** de Lambert: conserva las superficies (no las formas), así que sirve para medir áreas. Comparemos con lo que pasa si medimos en grados.

In [ ]:
area_grados = mundo.area   # ¿qué unidad tiene esto?
area_grados.head(3)

In [ ]:
# Escriban su código acá

### 1.4 Predecir: la capa de provincias del IGN

Leemos la capa de provincias de los videos y miramos el mapa **antes** de seguir.

In [ ]:
provincias = gpd.read_file(DATOS / "provincia.zip")
print(provincias.shape)
print(provincias.crs)
provincias[["in1", "nam"]].head(3)

In [ ]:
ax = provincias.plot(figsize=(4, 6), edgecolor="white", linewidth=0.3)
ax.set_title("La capa completa");

La provincia de Tierra del Fuego, Antártida e Islas del Atlántico Sur incluye el sector antártico, que llega hasta el polo. Para hacer mapas del país continental queremos sacarlo.

✍️ **Predigan:** si con `.cx` pedimos el rectángulo que va de −75 a −53 de longitud y de −56 a −21 de latitud (el continente más las Malvinas), ¿cuántas filas devuelve? ¿Desaparece la Antártida?

In [ ]:
rectangulo = (-75, -56, -53, -21)   # (xmin, ymin, xmax, ymax) = (lon mín, lat mín, lon máx, lat máx)
xmin, ymin, xmax, ymax = rectangulo

con_cx = provincias.cx[xmin:xmax, ymin:ymax]
print("Filas con .cx:  ", len(con_cx), "| latitud mínima:", con_cx.total_bounds[1].round(1))

continente = provincias.clip(rectangulo)
print("Filas con .clip:", len(continente), "| latitud mínima:", continente.total_bounds[1].round(1))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 6))
con_cx.plot(ax=axs[0], edgecolor="white", linewidth=0.3)
axs[0].set_title(".cx: selecciona filas")
continente.plot(ax=axs[1], edgecolor="white", linewidth=0.3)
axs[1].set_title(".clip: recorta geometrías")
for ax in axs:
    ax.set_axis_off()

---
## 2. Del censo al mapa

### 2.1 Un `merge` que no funciona

La clave de unión es el **código INDEC** de la provincia (dos dígitos). En la capa del IGN está en la columna `in1`. No usamos el nombre: tiene tildes, abreviaturas y variantes (por ejemplo, Tierra del Fuego con o sin «Antártida e Islas del Atlántico Sur»).

In [ ]:
pob_sin_dtype = pd.read_csv(DATOS / "poblacion_censos.csv")
print(pob_sin_dtype.head(3))
print("código en el CSV:", pob_sin_dtype["codigo"].dtype, "| in1 en la capa:", continente["in1"].dtype)

In [ ]:
continente.merge(pob_sin_dtype, left_on="in1", right_on="codigo")

`read_csv` vio solo dígitos, convirtió la columna en número y **borró el cero de adelante**: la Ciudad de Buenos Aires (`02`) quedó como `2`. Esta vez el error es ruidoso. La solución es leer los códigos como texto desde el principio:

In [ ]:
pob = pd.read_csv(DATOS / "poblacion_censos.csv", dtype={"codigo": str})
pob.head(3)

### 2.2 Un `merge` verificado

- `how="left"`: conservamos todas las provincias, aunque alguna no encuentre su dato.
- `validate="one_to_one"`: si una clave estuviera repetida, da error en vez de duplicar filas sin avisar (el `merge 1:1` de Stata).
- `indicator=True`: agrega la columna `_merge`, que dice de qué tabla vino cada fila (el `_merge` de Stata).
- El GeoDataFrame va **a la izquierda**, así el resultado sigue siendo un GeoDataFrame.

In [ ]:
mapa = continente[["in1", "nam", "geometry"]].merge(
    pob, left_on="in1", right_on="codigo",
    how="left", validate="one_to_one", indicator=True,
)
print(type(mapa).__name__, mapa.shape)
mapa["_merge"].value_counts()

In [ ]:
# Verificaciones que conviene escribir siempre después de un merge
assert len(mapa) == len(continente), "El merge cambió la cantidad de filas"
assert (mapa["_merge"] == "both").all(), "Hay provincias sin dato o datos sin provincia"
mapa = mapa.drop(columns=["_merge", "codigo"])

### 2.3 Dos maneras de medir el crecimiento

In [ ]:
mapa["var_abs"] = mapa["pob_2022"] - mapa["pob_2010"]
mapa["var_rel"] = (mapa["pob_2022"] / mapa["pob_2010"] - 1) * 100

crecimiento_pais = (pob["pob_2022"].sum() / pob["pob_2010"].sum() - 1) * 100
print(f"Crecimiento del país 2010-2022: {crecimiento_pais:.1f} %")

columnas = ["nam", "var_abs", "var_rel"]
display(mapa.nlargest(3, "var_abs")[columnas])
display(mapa.nlargest(3, "var_rel")[columnas])

### 2.4 Un mapa coroplético y el problema de la escala

✍️ **Predigan:** si pintamos la población de 2022 con una escala continua, ¿qué se va a ver?

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 7))
estilo = dict(cmap="viridis", edgecolor="white", linewidth=0.3)

mapa.plot(column="pob_2022", ax=axs[0], legend=True,
          legend_kwds={"label": "habitantes", "shrink": 0.6}, **estilo)
axs[0].set_title("Escala continua")

mapa.plot(column="pob_2022", ax=axs[1], scheme="quantiles", k=5, legend=True,
          legend_kwds={"title": "habitantes (quintiles)", "loc": "lower right", "fontsize": 7}, **estilo)
axs[1].set_title("Quintiles")

for ax in axs:
    ax.set_axis_off()

In [ ]:
# ¿Cómo quedaron las clases?
mapclassify.Quantiles(mapa["pob_2022"], k=5)

### 2.5 Ejercicio (5 minutos, cada uno en su notebook)

Hagan un mapa de la variación relativa (`var_rel`) con una **paleta divergente** (por ejemplo `"RdBu_r"`) centrada en el crecimiento del país, para que se vea qué provincias crecieron más y cuáles menos que el promedio.

*Ayuda:* `matplotlib.colors.TwoSlopeNorm(vmin=..., vcenter=..., vmax=...)` define una escala con un centro, y se le pasa a `plot` con el argumento `norm`.

In [ ]:
# Escriban su código acá

### 2.6 Densidad: ¿en qué sistema de coordenadas medimos?

Para la densidad necesitamos el área en km². Comparemos tres maneras de calcularla para tres provincias, una del norte, una del centro y una del sur.

In [ ]:
tres = mapa[mapa["nam"].isin(["Jujuy", "Buenos Aires", "Santa Cruz"])]

areas = pd.DataFrame({
    "provincia": tres["nam"],
    "grados2": tres.area,                                   # mal: grados cuadrados
    "mercator_km2": tres.to_crs("EPSG:3857").area / 1e6,     # en metros, pero no conserva áreas
    "igual_area_km2": tres.to_crs("ESRI:102033").area / 1e6, # Albers de igual área para Sudamérica
})
areas["inflado_mercator_%"] = (areas["mercator_km2"] / areas["igual_area_km2"] - 1) * 100
areas.round(1)

✍️ **Antes de calcular la densidad:** en `continente`, la provincia de Tierra del Fuego incluye la isla Grande, Isla de los Estados y las Malvinas, y no la Antártida. ¿Qué superficie usarían? No hay una respuesta automática: hay que elegir y decirlo en una nota del mapa.

In [ ]:
mapa["area_km2"] = mapa.to_crs("ESRI:102033").area / 1e6
mapa["densidad"] = mapa["pob_2022"] / mapa["area_km2"]
mapa.sort_values("densidad", ascending=False)[["nam", "area_km2", "densidad"]].head(4).round(1)

La Ciudad de Buenos Aires tiene más de 15.000 hab/km²; la provincia que le sigue, menos de 80. Con una escala lineal, el mapa sería un punto amarillo y todo lo demás violeta. Dos opciones válidas: una **escala logarítmica** o una clasificación por cuantiles. Excluir la Ciudad también vale, **si se dice**.

In [ ]:
from matplotlib.colors import LogNorm

ax = mapa.plot(
    column="densidad", cmap="viridis",
    norm=LogNorm(vmin=mapa["densidad"].min(), vmax=mapa["densidad"].max()),
    edgecolor="white", linewidth=0.3, figsize=(5, 8),
    legend=True, legend_kwds={"label": "hab/km² (escala logarítmica)", "shrink": 0.6},
)
ax.set_title("Densidad de población, 2022")
ax.set_axis_off()

### 2.7 Faltantes: lo que el mapa no muestra

Simulemos que no tenemos el dato de tres provincias.

In [ ]:
con_faltantes = mapa.copy()
sin_dato = con_faltantes["in1"].isin(["22", "42", "58"])   # Chaco, La Pampa, Neuquén
con_faltantes.loc[sin_dato, "var_rel"] = np.nan

fig, axs = plt.subplots(1, 2, figsize=(9, 7))
con_faltantes.plot(column="var_rel", ax=axs[0], cmap="viridis", legend=True,
                   legend_kwds={"shrink": 0.5})
axs[0].set_title("Sin missing_kwds")
con_faltantes.plot(column="var_rel", ax=axs[1], cmap="viridis", legend=True,
                   legend_kwds={"shrink": 0.5},
                   missing_kwds={"color": "lightgrey", "hatch": "///", "label": "sin dato"})
axs[1].set_title("Con missing_kwds")
for ax in axs:
    ax.set_axis_off()

### 2.8 Un mapa terminado (y reutilizable)

Esta función junta la lista de control de un mapa para un informe o una tesis: proyección adecuada, leyenda con unidad, faltantes visibles, título y fuente. Pueden copiarla para el TP Final y adaptarla.

In [ ]:
def mapa_coropletico(gdf, columna, titulo, etiqueta, fuente, ax=None, crs="ESRI:102033", **kwargs):
    """Dibuja un mapa coroplético con la lista de control mínima.

    gdf: GeoDataFrame con la columna a mapear.
    etiqueta: qué es la variable y en qué unidad está (va en la leyenda).
    fuente: fuente de los datos y de los límites (va al pie).
    kwargs: se pasan a GeoDataFrame.plot (cmap, scheme, k, norm, vmin, vmax, ...).
    Ojo: con barra de colores (sin scheme), «sin dato» no aparece en la leyenda; aclárenlo en la fuente.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 8))
    opciones = {
        "cmap": "viridis", "edgecolor": "white", "linewidth": 0.3, "legend": True,
        "missing_kwds": {"color": "lightgrey", "hatch": "///", "label": "sin dato"},
    }
    if "scheme" in kwargs:   # leyenda por clases
        opciones["legend_kwds"] = {"title": etiqueta, "loc": "lower right", "fontsize": 8}
    else:                    # barra de colores
        opciones["legend_kwds"] = {"label": etiqueta, "shrink": 0.6}
    opciones.update(kwargs)
    gdf.to_crs(crs).plot(column=columna, ax=ax, **opciones)
    ax.set_title(titulo, fontsize=12)
    ax.set_axis_off()
    ax.annotate(fuente, xy=(0, -0.02), xycoords="axes fraction", fontsize=7, va="top", color="dimgrey")
    return ax

In [ ]:
mapa_coropletico(
    mapa, "var_rel",
    titulo="Crecimiento de la población por provincia, 2010-2022",
    etiqueta="variación (%)",
    fuente=("Fuente: INDEC, censos 2010 y 2022. Límites: IGN, sin el sector antártico.\n"
            "Proyección: Albers de igual área para Sudamérica (ESRI:102033)."),
    scheme="natural_breaks", k=5,
);

---
## 3. Desafío en salas: ¿en qué provincia está cada localidad?

**Equipos de 3, 35 minutos, sin asistentes de IA.** Al volver, un equipo comparte su respuesta a 3.5.

**La situación.** Para su tesis necesitan contar localidades y población por provincia. Consiguieron `localidades_ne.csv`, con 158 localidades argentinas de Natural Earth: nombre, provincia, tipo, población y coordenadas. ¿Se puede confiar en la columna `provincia_ne`?

**Lo que tienen que entregar al volver:** la cantidad de localidades en las que la columna `provincia_ne` y la ubicación no coinciden, una explicación de por qué, y un mapa que lo muestre.

### 3.1 Mirar los datos

Lean el archivo y miren los valores distintos de `provincia_ne`. ¿Cuántos hay? ¿Qué les llama la atención?

In [ ]:
# Escriban su código acá

### 3.2 De columnas a puntos

Construyan un GeoDataFrame `localidades` con una geometría de puntos a partir de `latitud` y `longitud`, con CRS `EPSG:4326`. Después **verifiquen** con un `assert` que todos los puntos caen dentro del rectángulo de `continente` (`continente.total_bounds`).

*Ayuda:* `gpd.points_from_xy(x, y)`. ¿Cuál es la `x`?

In [ ]:
# Escriban su código acá

### 3.3 Unir por ubicación

Asignen a cada localidad la provincia del IGN en la que cae, con `sjoin` y `predicate="within"`. Usen `how="left"` para no perder localidades y quédense solo con las columnas `in1` y `nam` de `continente`.

1. ¿El resultado tiene la misma cantidad de filas que `localidades`? ¿Por qué podría tener más?
2. ¿Quedó alguna localidad sin provincia? ¿Cuál, y por qué?

In [ ]:
# Escriban su código acá

In [ ]:
# (Opcional) Asignen las que quedaron afuera a la provincia más cercana con sjoin_nearest

### 3.4 Normalizar la columna `provincia_ne` con expresiones regulares

Para comparar la columna con el resultado del `sjoin` necesitamos pasar `provincia_ne` al código INDEC. Completen la función `codigo_desde_texto`, que recibe un texto como `"RRo Negro"` y devuelve `"62"`.

Una idea: a partir del nombre oficial de cada provincia, armen un patrón donde **cada letra con tilde se reemplaza por un punto** (`"Río Negro"` → `^r.o negro$`). Así coinciden `Río Negro`, `Rio Negro` y `RRo Negro`. Dos jurisdicciones necesitan un patrón escrito a mano. Si un texto coincide con **cero o con más de un** patrón, la función devuelve `None`: no adivinamos.

In [ ]:
import re

oficiales = continente.set_index("in1")["nam"].to_dict()
oficiales

In [ ]:
def codigo_desde_texto(texto):
    """Devuelve el código INDEC de la provincia, o None si no hay exactamente una coincidencia."""
    # Completen
    return None

Casos que tienen que funcionar (no los modifiquen):

In [ ]:
casos = {
    "Río Negro": "62", "Rio Negro": "62", "RRo Negro": "62",
    "Neuqutn": "58", "CRrdoba": "14", "Tucumtn": "90",
    "Buenos Aires": "06", "Ciudad de Buenos Aires": "02",
    "Tierra del Fuego": "94", "Entre Ríos": "30",
    "Montevideo": None, "": None,
}
for texto, esperado in casos.items():
    obtenido = codigo_desde_texto(texto)
    assert obtenido == esperado, f"{texto!r}: se esperaba {esperado!r} y se obtuvo {obtenido!r}"
print("Los", len(casos), "casos pasan")

In [ ]:
# Apliquen la función a toda la columna y verifiquen que no quede ningún None

### 3.5 Comparar las dos fuentes

1. ¿En cuántas localidades la provincia de la columna (`codigo_ne`) no coincide con la del `sjoin` (`in1`)?
2. Armen una tabla cruzada de los desacuerdos (`pd.crosstab`): provincia según la columna contra provincia según la ubicación.
3. Hagan un mapa: las provincias de fondo, los acuerdos en un color y los desacuerdos en otro.
4. **Discutan:** ¿cuál de las dos fuentes está mal? ¿Cómo lo saben? Busquen dos ejemplos que conozcan.

In [ ]:
# Escriban su código acá

In [ ]:
# Mapa de acuerdos y desacuerdos

### 3.6 Resumir y mapear

Con la provincia **según la ubicación**, cuenten las localidades por provincia y súmenles la población. Unan ese resumen a `mapa` y hagan un mapa con las provincias pintadas según la cantidad de localidades y los puntos con un tamaño proporcional a la población.

*Ayuda:* `groupby("in1").agg(localidades=("nombre", "size"), poblacion=("poblacion", "sum"))`. Para el tamaño de los puntos, `markersize=` acepta una serie.

In [ ]:
# Resumen por provincia, merge con mapa y mapa

### 3.7 Para los que terminan antes

Para cada localidad que no es capital, calculen la distancia en km a la **capital provincial más cercana** con `sjoin_nearest`. ¿Cuáles son las cinco más lejanas? ¿La capital más cercana es siempre la de su provincia?

In [ ]:
# Distancia a la capital provincial más cercana

---
## 4. Departamentos: códigos, `sjoin`, `dissolve` y mapas interactivos

### 4.1 La capa de departamentos y la codificación de los textos

In [ ]:
print(departamentos.shape, departamentos.crs)
departamentos[["in1", "nam", "fna"]].head()

La capa no trae el nombre de la provincia. Pero `in1` es el código INDEC del departamento, de **cinco dígitos**: los dos primeros son la provincia y los tres siguientes, el departamento. Lo separamos con una expresión regular con grupos con nombre, y verificamos que **todas** las filas coincidan con el patrón:

In [ ]:
partes = departamentos["in1"].str.extract(r"^(?P<cod_prov>\d{2})(?P<cod_depto>\d{3})$")
assert partes.notna().all().all(), "Hay códigos que no tienen cinco dígitos"

departamentos = departamentos.join(partes)
departamentos["cod_prov"].value_counts().head()

### 4.2 La otra manera: asignar la provincia por ubicación

Si la capa no tuviera códigos, podríamos asignar la provincia con un `sjoin`. Para eso necesitamos un **punto** de cada departamento: si usáramos el polígono, un departamento que toca dos provincias aparecería dos veces.

✍️ **Predigan:** ¿el centroide de un polígono siempre está adentro del polígono?

In [ ]:
deptos_proj = departamentos.to_crs("ESRI:102033")
centroide_afuera = ~deptos_proj.centroid.within(deptos_proj.geometry)
print("Departamentos cuyo centroide cae afuera:", centroide_afuera.sum())
departamentos.loc[centroide_afuera, ["in1", "nam"]]

In [ ]:
# representative_point() garantiza un punto dentro del polígono
puntos = departamentos[["in1", "cod_prov", "nam"]].copy()
puntos = gpd.GeoDataFrame(puntos, geometry=departamentos.representative_point(), crs=departamentos.crs)

por_ubicacion = puntos.sjoin(
    provincias[["in1", "geometry"]].rename(columns={"in1": "prov_ubicacion"}),
    how="left", predicate="within",
)
print("filas:", len(puntos), "->", len(por_ubicacion))
print("sin provincia:", por_ubicacion["prov_ubicacion"].isna().sum())
desacuerdo = por_ubicacion["prov_ubicacion"] != por_ubicacion["cod_prov"]
por_ubicacion.loc[desacuerdo, ["in1", "nam", "cod_prov", "prov_ubicacion"]]

### 4.3 De departamentos a provincias: `dissolve`

`dissolve` es un `groupby` para geometrías: une los polígonos de cada grupo y agrega las columnas con la función que le pidamos. Sirve para pasar de una unidad chica a una grande (de radios censales a departamentos, de departamentos a provincias, de municipios a regiones).

In [ ]:
# Para que sea más rápido, simplificamos las geometrías (tolerancia de 0,01 grados, ~1 km)
deptos_simple = departamentos[["cod_prov", "geometry"]].copy()
deptos_simple["geometry"] = deptos_simple.simplify(0.01)
deptos_simple["n_departamentos"] = 1

desde_deptos = deptos_simple.dissolve(by="cod_prov", aggfunc={"n_departamentos": "sum"})
print(desde_deptos.shape)
desde_deptos.head(3)

In [ ]:
# Verificación: ¿las provincias armadas con departamentos tienen la misma superficie que la capa de provincias?
comparacion = pd.DataFrame({
    "desde_deptos_km2": desde_deptos.to_crs("ESRI:102033").area / 1e6,
    "capa_provincias_km2": provincias.set_index("in1").to_crs("ESRI:102033").area / 1e6,
})
comparacion["dif_%"] = (comparacion["desde_deptos_km2"] / comparacion["capa_provincias_km2"] - 1) * 100
comparacion.round(1).sort_values("dif_%")

### 4.4 Comparar mapas: la misma escala

Los departamentos de una provincia, coloreados por su superficie, en dos provincias lado a lado. ¿Qué pasa con la escala de colores si no la fijamos?

In [ ]:
deptos_proj["area_km2"] = deptos_proj.area / 1e6

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
vmax = deptos_proj.loc[deptos_proj["cod_prov"].isin(["14", "82"]), "area_km2"].max()
for ax, codigo, nombre in zip(axs, ["14", "82"], ["Córdoba", "Santa Fe"]):
    deptos_proj[deptos_proj["cod_prov"] == codigo].plot(
        column="area_km2", ax=ax, cmap="viridis", vmin=0, vmax=vmax,   # misma escala en los dos paneles
        edgecolor="white", linewidth=0.3, legend=True, legend_kwds={"label": "km²", "shrink": 0.7},
    )
    ax.set_title(nombre)
    ax.set_axis_off()

### 4.5 Mapas interactivos: `explore()`

`explore()` arma un mapa web (con la librería Folium) donde se puede hacer zoom y pasar el cursor sobre cada polígono. Necesita internet para el mapa de fondo.

In [ ]:
# Para un mapa web conviene simplificar: con la geometría completa del IGN, el mapa pesa unos 36 MB
mapa_web = mapa.assign(geometry=mapa.simplify(0.01))   # tolerancia de 0,01 grados (~1 km)

mapa_web.explore(
    column="var_rel",
    cmap="viridis",
    tooltip=["nam", "pob_2010", "pob_2022", "var_rel"],
    legend_kwds={"caption": "Variación de la población 2010-2022 (%)"},
)

In [ ]:
# Se puede guardar como HTML y abrir con cualquier navegador
m = mapa_web.explore(column="var_rel", cmap="viridis", tooltip=["nam", "var_rel"])
m.save("mapa_crecimiento.html")

**Con un widget.** Un selector de provincia que muestra sus departamentos (lo vimos en la clase 4):

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

opciones = sorted((nombre, codigo) for codigo, nombre in oficiales.items())

@interact(codigo=widgets.Dropdown(options=opciones, description="Provincia"))
def mostrar_departamentos(codigo):
    seleccion = departamentos[departamentos["cod_prov"] == codigo]
    ax = seleccion.plot(cmap="tab20", edgecolor="white", linewidth=0.4, figsize=(6, 6))
    ax.set_title(f"{oficiales[codigo]}: {len(seleccion)} departamentos")
    ax.set_axis_off()
    plt.show()

### 4.6 Otras herramientas (para el TP)

- **`contextily`**: agrega un mapa de fondo (calles, relieve) a un mapa estático.
- **`plotly.express.choropleth`**: mapas interactivos con la lógica de los gráficos de Plotly.
- **`folium`**: control fino de un mapa interactivo (capas, marcadores, popups).
- **QGIS**: para mirar una capa con el mouse o hacer cartografía de publicación.

Antes de sumar una librería, pregúntense: ¿qué puedo hacer con ella que no pueda hacer con lo que ya tengo? ¿Se instala con un comando? ¿El resultado depende de internet o de una clave? Esas respuestas deciden si su trabajo es reproducible.

---
## 5. Presentación del TP Final

La consigna completa está en `PythEcon_tp6_final.ipynb`. En resumen:

- Es **individual**, con **sus propios datos** (idealmente, los de su tesis).
- Tiene que combinar: obtención reproducible de datos, **expresiones regulares**, **pandas** (uniones verificadas, agregaciones, cambios de forma), **gráficos** y **mapas**.
- Se entrega en un repositorio de GitHub que **otra persona pueda ejecutar**.
- Se defiende en una **exposición individual** con la docente como evaluadora, que incluye una modificación del código en vivo.

### Antes de irse: cinco preguntas para cualquier mapa

1. **¿Qué mide?** Variable, unidad y período. ¿Total o tasa? Si es tasa, ¿cuál es el denominador?
2. **¿Con qué geometrías?** Fuente, nivel y qué incluye cada polígono.
3. **¿En qué sistema de coordenadas?** Si midieron áreas o distancias, ¿en una proyección adecuada?
4. **¿Con qué escala y qué colores?** Esquema de clasificación, paleta, misma escala entre paneles.
5. **¿Qué falta?** ¿Se ven los faltantes? ¿Se verificaron las uniones? ¿Se explican las limitaciones?